This notebook uses Decision Tree Anaysis to explore the relationship between homes flagged for a high risk of loan default and various risk characteristics.

In [7]:
import seaborn as sns
import os
import json
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (20, 6)
plt.rcParams['font.size'] = 14
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (accuracy_score, 
                             classification_report, 
                             confusion_matrix, auc, roc_curve
                            )

loan = pd.read_csv("C:/Users/jacks/OneDrive/Documents/Loan/Loan Prediction.csv")

In [11]:
loan.head()


,Id,Income,Age,Experience,Married/Single,House_Ownership,Car_Ownership,Profession,CITY,STATE,CURRENT_JOB_YRS,CURRENT_HOUSE_YRS,Risk_Flag
0,1,1303834,23,3,single,rented,no,Mechanical_engineer,Rewa,Madhya_Pradesh,3,13,0
1,2,7574516,40,10,single,rented,no,Software_Developer,Parbhani,Maharashtra,9,13,0
2,3,3991815,66,4,married,rented,no,Technical_writer,Alappuzha,Kerala,4,10,0
3,4,6256451,41,2,single,rented,yes,Software_Developer,Bhubaneswar,Odisha,2,12,1
4,5,5768871,47,11,single,rented,no,Civil_servant,Tiruchirappalli[10],Tamil_Nadu,3,14,1


In [8]:
loan['Married'] = loan['Married/Single'].str.contains('married').astype(int)
loan['Car_Ownership'] = loan.Car_Ownership.str.contains('yes').astype(int)

In [10]:
from sklearn.model_selection import train_test_split
non_num = ['Id', 'Married/Single',
       'House_Ownership', 'Profession', 'CITY', 'STATE', 'Married', 'norentnoown', 'rent',
       'own']
xt = loan.copy().drop(non_num, axis=1)
model = DecisionTreeClassifier(criterion='entropy')
x_train, x_test, y_train, y_test = train_test_split(xt.drop(['Risk_Flag'], axis=1),xt.Risk_Flag, test_size=.20)

model.fit(x_train, y_train)

DecisionTreeClassifier(criterion='entropy')

In [87]:
ttest_predictions = model.predict(x_ttest)
accuracy_score(y_ttest, ttest_predictions)

0.8817261904761905

In [88]:
confusion_matrix(y_ttest, ttest_predictions)

array([[40983,  3198],
       [ 2763,  3456]], dtype=int64)

In [89]:
print(classification_report(y_ttest, ttest_predictions))

              precision    recall  f1-score   support

           0       0.94      0.93      0.93     44181
           1       0.52      0.56      0.54      6219

    accuracy                           0.88     50400
   macro avg       0.73      0.74      0.73     50400
weighted avg       0.89      0.88      0.88     50400



In [90]:
list(zip(xt.drop(['Risk_Flag'], axis=1).columns, model.feature_importances_))

[('Income', 0.44610608728659035),
 ('Age', 0.22740317931584508),
 ('Experience', 0.09818689357786814),
 ('Car_Ownership', 0.03035945885808729),
 ('CURRENT_JOB_YRS', 0.09830612658244416),
 ('CURRENT_HOUSE_YRS', 0.0996382543791649)]

In [21]:
xt.Risk_Flag.value_counts()

0    221004
1     30996
Name: Risk_Flag, dtype: int64

Here we see that with initial analysis, income has the highest impact on an individual's likelihood to be flagged for being at risk to default on a loan. With a model importance score of .44%, income appears to explain 44% of the variance in the model's prediction of whther or not an individual is flagged as a lending risk. However, when expamining the classification report, the algorithm seems to correctly predict the absence of a flag 92% of the time, and the presence of the flag only 52% of the time. This is likely because entries with the risk flag of 1 are over represented in the sample, representing 221,004 entries while only 30,996 entries represent flagged profiles. SMOTE analyses will be used below to aid in the precision of these estimates, hopefully increasing the accuracy of the model in correctly identifying flagged entries. In order to do this nearest neighbors analysis was used to generate synthetic flagged entries. 

In [12]:

#xt = loan.copy().drop(non_num2, axis=1)
#model = DecisionTreeClassifier(criterion='entropy')
#x_ttrain, x_ttest, y_ttrain, y_ttest = train_test_split(xt.drop(['Risk_Flag'], axis=1),xt.Risk_Flag, test_size=.20)




smote=SMOTE(sampling_strategy='minority') 
x,y=smote.fit_resample(xt.drop(['Risk_Flag'], axis=1),xt.Risk_Flag)
#y.value_counts()
x_strain, x_stest, y_strain, y_stest = train_test_split(x,y, test_size=.20)
model.fit(x_strain, y_strain)

DecisionTreeClassifier(criterion='entropy')

In [13]:
stest_predictions = model.predict(x_stest)
accuracy_score(y_stest, stest_predictions)

0.9032714191986607

In [16]:
confusion_matrix(y_stest, stest_predictions)

array([[37990,  6248],
       [ 2303, 41861]], dtype=int64)

In [14]:
list(zip(xt.drop(['Risk_Flag'], axis=1).columns, model.feature_importances_))

[('Income', 0.4463525372693866),
 ('Age', 0.21673212372020112),
 ('Experience', 0.12067203509335911),
 ('Car_Ownership', 0.006286546038543565),
 ('CURRENT_JOB_YRS', 0.13240808642451868),
 ('CURRENT_HOUSE_YRS', 0.07754867145399078)]

In [19]:
y.value_counts()

0    221004
1    221004
Name: Risk_Flag, dtype: int64

In [15]:
print(classification_report(y_stest, stest_predictions))

              precision    recall  f1-score   support

           0       0.94      0.86      0.90     44238
           1       0.87      0.95      0.91     44164

    accuracy                           0.90     88402
   macro avg       0.91      0.90      0.90     88402
weighted avg       0.91      0.90      0.90     88402



With the synthetic entries, the precision of flagged estimates rose from 52% to 87% and an f1 score of 91%. Though the smote analysis decreased the f1 score of non-flagged entries, this is a reasonable trade off for the increased accuracy of flagged entry prediction facilitated by SMOTE analysis. While income remains the most important feature in the analysis prediction, car ownership fell significantly as well as tenure in current home. At the same time job experience and years spent working at the current job increased in prediction importance. The model findings show trends that the average person may infer: those with steady amd predicatble livelihoods are less likely to default on their loans. The higher one's income, the more likely they will have the resources to handle unexpected costs while handling their debt responsibilities. 